# Run Easy_HCR and check probe pair quality

This notebook is used to run Easy_HCR and then check the quality of the probe pairs (PP) generated.
In order to run this notebook you can just **click on the double arrow button** at the top left of the notebook. This will then **run automatically everything** and will **stop** at certain steps so you can **make a decision** on some parameters. In that case you never need to run a cell (grey rectangle in the code) yourself, they are executed automatically one by one.

You can also **run it step by step** if you **select the cell and click on the single arrow** on the top left of the notebook. In that case you need to **redo this for every cell of the notebook**.

For more information on Easy_HCR refer to the [website](https://gitlab.com/NCDRlab/easy_hcr)

In [ ]:
import pandas as pd
import numpy as np
import csv
import subprocess
import gzip
import math
from insitu_probe_generator.maker37cb import maker
from insitu_probe_generator.start import start
from Bio import SearchIO
from Bio import SeqIO
from Bio.Seq import Seq
from Bio.SeqRecord import SeqRecord
from Bio.Blast import NCBIXML
from pathlib import Path

In [ ]:
# Set the quality control parameters for the probeset
# if you do not want to remove the GC's you can set 'remove_gc' to 'False'

remove_gc = True
probepair_number_cutoff = 33

stringent_transcriptome = input("Do you want to do a stringent blast search against the transcriptome? (Y/N): \nThis will make the blast search more strict to find hits in the transcriptome so you will find less problematic probes.\nThe default value is N but if too many probes are found I suggest to set this parameter to Y.").strip().upper()
if stringent_transcriptome == "Y":
    stringent_blast_search = True
    print("\nThe transcriptome search will be stringent.\n")
else:
    stringent_blast_search = False
    print("\nThe transcriptome search will not be stringent.\n")
        
stringent_genome = input("Do you want to do a stringent blast search against the Octopus genomes? (Y/N): \nThis will make the blast search more strict to find hits in the genomes of O. vulgaris and sinensis so you will find less problematic probes.\nThe default value is N but if too many probes are found I suggest to set this parameter to Y.").strip().upper()
if stringent_genome == "Y":
    stringent_selection = True
    print("\nThe blast search against the genome will be stringent.")
else:
    stringent_selection = False
    print("\nThe blast search against the genome will not be stringent")

In [ ]:
# Create an output directory

Path("output").mkdir(parents=True, exist_ok=True)

In [ ]:
# Run the probe generator

strt = start()
name,fullseq,amplifier,pause,choose,polyAT,polyCG,BlastProbes,db,dropout,show,report,maxprobe,numbr = strt[0],strt[1],strt[2],strt[3],strt[4],strt[5],strt[6],strt[7],strt[8],strt[9],strt[10],strt[11],strt[12],strt[13]
string_output = maker(name,fullseq,amplifier,pause,choose,polyAT,polyCG,BlastProbes,db,dropout,show,report,maxprobe,numbr)
gene_name = name
amplifier = amplifier

In [ ]:
# Save the generated probes as CSV file

csv_path = f"output/{name}_probes.csv"

probe_file = open(csv_path, "w")
n = probe_file.write(string_output)
probe_file.close()

# Read the csv back in as table

df = pd.read_csv(
    f"output/{name}_probes.csv",
    index_col=0,
)
print("Gene: ", gene_name)
print("Amplifier: ", amplifier)
print("Probes:")
df.head()

In [ ]:
# Generate probe identifiers

probe_identifiers = [gene_name + "_PP_" + str(n + 1) for n,_ in enumerate(df.iterrows())]
probe_identifiers = [f"{gene_name}_{amplifier}_PP_{str(n + 1)}" for n,_ in enumerate(df.iterrows())]
df.index = probe_identifiers
df.head()

In [ ]:
# Check GC-ending probes for later
# GC and the end for the forst probe and GC at the start for the second

df["GC_QC"] = df["Probe"].str.endswith("GC") | df["Probe"].str.endswith("CG") | df["Probe.1"].str.startswith("GC") | df["Probe.1"].str.startswith("CG")
print("Number of probe pairs:", len(df))
print(f"{len(df.loc[df.GC_QC == True])} GC probe pairs")
print(f"Kept {len(df.loc[df.GC_QC != True])} no-GC probe pairs")
df.head()


In [ ]:
# Generate BLAST sequences

df["blast"] = df["Probe"] + "NN" + df["Probe.1"]
df.head()

In [ ]:
# Create a simple dataframes for BLAST

df_blast = pd.DataFrame(df.index.values, columns=["Identifier"])
df_blast["Probe"] = df.blast.values
df_blast.to_csv(f"output/{gene_name}_probes.txt",sep='\t',index=False,header=False, quoting=csv.QUOTE_NONE)
df_blast.head()

In [ ]:
# Generate a list of SeqRecord to write a fasta file

probe_sequences = [SeqRecord(Seq(probe), id=id) for probe, id in zip(df_blast.Probe.values, df_blast.Identifier.values)]

In [ ]:
# Create a fasta file containing the probes' sequences in the output folder (create this output folder in the same folder as the probe generator)

fasta_path = f"output/{gene_name}_to_blast_against_vulgaris_transcriptome.fasta"

with open(fasta_path, "w") as output_handle:
    for record in probe_sequences:
        SeqIO.write(record, output_handle, "fasta")

fasta_path

In [ ]:
# Extract the position (chromosome, start position and end position) of the transcript of interest in the O. vulgaris genome.
# This step is informative and is needed for the mapping against the O. vulgaris genome.

gtf_path = "necessary_files/xcOctVulg1.1_MT_fixed_mergedPeaks.gtf.gz"
target_length = len(fullseq)

# The length of the transcript in the transcript FASTA file correspond to the cumulative size of each exon of a transcript in the annotation GTF file.

# We first create a dictionnary to store the cumulative size of the exons of each transcript.
transcript_data = {}

with gzip.open(gtf_path, "rt") as f:
    for line in f:
        if line.startswith("#"): continue
        columns = line.split("\t")
        
        if len(columns) > 8 and columns[2] == "exon" and gene_name in line:
            attributes = columns[8]
            t_id = None
            if 'transcript_id "' in attributes:
                t_id = attributes.split('transcript_id "')[1].split('"')[0]
            else:
                t_id = gene_name

            exon_start = int(columns[3])
            exon_end = int(columns[4])
            exon_len = (exon_end - exon_start) + 1
            
            if t_id not in transcript_data:
                transcript_data[t_id] = {
                    "chrom": columns[0],
                    "total_len": 0,
                    "min_start": exon_start,
                    "max_end": exon_end
                }
            
            transcript_data[t_id]["total_len"] += exon_len
            transcript_data[t_id]["min_start"] = min(transcript_data[t_id]["min_start"], exon_start)
            transcript_data[t_id]["max_end"] = max(transcript_data[t_id]["max_end"], exon_end)

# Then, we search the transcript with the closest length to our sequence of interest.
best_chrom, best_start, best_end = None, None, None
best_t_id = None
min_diff = float('inf')

for t_id, data in transcript_data.items():
    diff = abs(data["total_len"] - target_length)
    if diff < min_diff:
        min_diff = diff
        best_t_id = t_id
        best_chrom = data["chrom"]
        best_start = data["min_start"]
        best_end = data["max_end"]

if best_chrom:
    found_len = transcript_data[best_t_id]["total_len"]
    print(f"Length of the transcript of interest: {target_length} bp")
    print(f"Best matching transcript: {best_t_id}")
    print(f"Position on O. vulgaris: {best_chrom}:{best_start}-{best_end}")
else:
    print(f"No exons found for '{gene_name}' in the GTF.")

In [ ]:
# Create the output path for the blast against O. vulgaris transcriptome
blast_path = f"output/{gene_name}_blast_against_Ovul_transcriptome.xml"

In [ ]:
# Create the blast command depending on the chosen mode (stringent or not)

# Define name of the transcriptome database (Don't forget to first run the custom_database_creation.ipynb script on the file)
db_path = "xcOctVulg1.1_MT_mergedPeaks_CDS.fa"

if stringent_blast_search == True:
    cmd_blastn = [
        "blastn",
        "-query", fasta_path,
        "-db", db_path,
        "-evalue", "0.05",
        "-outfmt", "5",
        "-penalty", "-3",
        "-reward", "2",
        "-gapopen", "5",
        "-gapextend", "2",
        "-word_size", "11",
        "-out", blast_path
    ]
else:
    cmd_blastn = [
        "blastn",
        "-query", fasta_path,
        "-db", db_path,
        "-evalue", "0.2",
        "-outfmt", "5",
        "-word_size", "11",
        "-out", blast_path
    ]

'output/OvTH_to_blast.fasta'

In [ ]:
# Run blast

try:
    print(f"Blasting the probes against the O. vulgaris transcriptome...")
    result = subprocess.run(cmd_blastn, capture_output=True, text=True, check=True)
    print("Finished!")
    print(result.stdout) # Print output of blast search
except subprocess.CalledProcessError as e:
    print(f"Error during the blast search : {e.stderr}")

In [ ]:
# Read the xml file containing the results
result_handle = open(blast_path)
blast_records = list(NCBIXML.parse(result_handle))
result_handle.close()

In [ ]:
# Print a table listing the blast results grouped per probe pair
# You can check the transcript ids here.

queries = []
names = []

for record in blast_records:
    for alignment in record.alignments:
        queries.append(record.query.split(" ")[0])
        title = alignment.title
        name_parts = title.split("|")
        names.append(name_parts[-1])
    

df_blast_results = pd.DataFrame(queries, columns=["Probe"])
df_blast_results["Name"] = names
df_blast_results

In [ ]:
# Print a table that shows the number of hits per probe pair
df_hits = df_blast_results['Probe'].value_counts().reset_index()
df_hits.columns = ['Probe', 'n_hits']
df_hits['Probe'] = df_hits['Probe'].str.split(" ").str[0]
df_hits = df_hits.set_index('Probe')

df_hits

In [ ]:
## Removal of all probes with too many hits in the O. vulgaris transcriptome.
# You will be asked to choose a number for the threshold, I suggest to give the number of transcripts your gene of interest has.

hit_threshold = int(input("How many transcripts does your gene have? (number): \n This will remove the probe pairs which have more matches than your gene of interest.\n").strip())

to_exclude = df_hits.loc[df_hits.n_hits > hit_threshold].index.values.tolist()
print("Probes to remove:")
print(to_exclude)

blastn -out output/OvTH_blast_output.xml -outfmt 5 -query output/OvTH_to_blast.fasta -db input/vulgaristranscriptome.fasta -evalue 0.2 -word_size 11


In [ ]:
# If you need to manually remove additional probes you can paste the name between the brackets
# This prints the list of the selected probes to be removed

manual_exclude = [
    
]

probes_to_remove = to_exclude + manual_exclude
probes_to_remove

In [ ]:
# Remove unwanted probes

on_topic_indexes = [n for n, probe in enumerate(df.index.values) if probe not in probes_to_remove]
df_qc_blast = df.iloc[on_topic_indexes]
print(f"{len(on_topic_indexes)} probes kept:\n", df_qc_blast.index.unique().values)
df_qc_blast.head(10)

In [ ]:
# Remove probes ending in GC or CG

if remove_gc == True:
    orig_number = len(df_qc_blast)
    print("Previous number of probe pairs:", orig_number)
    df_qc_blast = df_qc_blast.loc[df_qc_blast.GC_QC != True]
    print(f"{orig_number - len(df_qc_blast)} probe pairs removed due to GC issues")
    print(f"Updated number of probe pairs:", len(df_qc_blast))
df_qc_blast.head(10)

In [ ]:
# If there are more than 50 probe pairs, keep only the odd rows, otherwise cut off at the specified amount
n_probes = len(df_qc_blast)
print("Total number of probes: ", n_probes)

if len(df_qc_blast.iloc[::2]) > 25:
    df_qc_blast = df_qc_blast.iloc[::2]
    print(f"Kept odd rows ({len(df_qc_blast)} probes)")

df_qc_blast = df_qc_blast.iloc[:probepair_number_cutoff]

new_length = len(df_qc_blast)
n_probes = new_length
print("Number of kept probes: ", n_probes)

In [ ]:
# Here you will be asked if you want to blast the remaining probes against the genomes of O. vulgaris and O. sinensis

blast_genomes = input("Do you want to blast the remaining probes against the genomes of O. vulgaris and O. sinensis? (Y/N): \nThis will blast the transcript of interest against the genomes to see which regions are problematic\nand then this will blast the remaining probes against the transcript of interest to see which probes align\nin the problematic regions of the transcript.\n").strip().upper()

In [ ]:
# Create a new fasta file containing the sequences of the remaining probes in the output folder
if blast_genomes == "Y":
    df_blast = pd.DataFrame(df_qc_blast.index.values, columns=["Identifier"])
    df_blast["Probe"] = df_qc_blast.blast.values
    df_blast.to_csv(f"output/{gene_name}_probes.txt",sep='\t',index=False,header=False, quoting=csv.QUOTE_NONE)
    probe_sequences = [SeqRecord(Seq(probe), id=id) for probe, id in zip(df_blast.Probe.values, df_blast.Identifier.values)]
    fasta_path = f"output/{gene_name}_to_blast_against_vulgaris_genome.fasta"
    with open(fasta_path, "w") as output_handle:
        for record in probe_sequences:
            SeqIO.write(record, output_handle, "fasta")

In [ ]:
# Create a fasta file containing the transcript sequence in order to blast it against the O. vulgaris genome
if blast_genomes == "Y":
    file_path = f"output/{gene_name}.fasta"
    with open(file_path, "w") as f:
        f.write(f">{gene_name}\n")
        f.write(f"{fullseq}\n")

In [ ]:
# Align the transcript of interest to the O. vulgaris genome for extra check
# Load database generated from O. vulgaris genome (Here again you need to make sure you ran the custom_database_creation.ipynb script on the file)
if blast_genomes == "Y":
    db_path = "xcOctVulg1.1_MT.fa.gz"

    # Define output file name
    blast_path = f"output/{gene_name}_blast_against_Ovul_genome.xml"

    # Create blast command
    cmd_blastn = [
        "blastn",
        "-query", file_path,
        "-db", db_path,
        "-task", "blastn",
        "-outfmt", "5",
        "-evalue", "0.05",
        "-out", blast_path
    ]

    # Run blast and save the results
    try:
        print(f"Blasting the transcript of interest against the O. vulgaris genome...")
        result = subprocess.run(cmd_blastn, capture_output=True, text=True, check=True)
        print("Finished!")
        print(result.stdout)
    except subprocess.CalledProcessError as e:
        print(f"Error during the blast search : {e.stderr}")

In [ ]:
# Check the results of this blast search
if blast_genomes == "Y":
    qresults = SearchIO.parse(blast_path, "blast-xml")

    # Generate a table of blast results grouped per probe pair # you can check the transcript ids here
    rows = []
    for qresult in qresults:
        for hit in qresult:
            for hsp in hit:
                identity = (hsp.ident_num / hsp.aln_span) * 100
                rows.append({
                    "target": hit.id,
                    "Identity": identity,
                    "Start_trans": hsp.query_start,
                    "End_trans": hsp.query_end,
                    "Start_gen": hsp.hit_start,
                    "End_gen": hsp.hit_end
                })

    df_all_hits = pd.DataFrame(rows)

In [ ]:
# Extract the problematic regions in a variable
if blast_genomes == "Y":
    df_all_hits['Aln_Length'] = (df_all_hits['End_trans'] - df_all_hits['Start_trans']).abs()

    if stringent_selection == True:
        min_length_danger = math.ceil(len(fullseq) / 15)
        min_identity_danger = 80
    else:
        min_length_danger = 1
        min_identity_danger = 1

    # On définit les "bad_regions" : hits qui ne sont pas sur notre gène cible
    mask_target = (
        (df_all_hits['target'] == best_chrom) & 
        (df_all_hits['Start_gen'] > best_start - 10) & 
        (df_all_hits['End_gen'] < best_end + 10)
    )

    df_bad_regions = df_all_hits[ (~mask_target) & (df_all_hits['Aln_Length'] >= min_length_danger) & (df_all_hits['Identity'] >= min_identity_danger) ].copy()
    df_unique = (df_bad_regions
                     .sort_values(by="Start_trans", ascending=False)
                     .drop_duplicates(subset=['Start_trans', 'End_trans']))
    print(f"\nTotal bad regions found in the transcript of interest: {len(df_unique)}")
    print("\n(Here you can check the mapping results of all the regions detected as bad in the transcript)\n")
    print(df_unique)

In [ ]:
# Align the remaining probes to the transcript of interest to eliminate the ones mapping in bad regions of the transcript
if blast_genomes == "Y":
    
    # Define the name of the output file
    probes_vs_trans_xml = f"output/{gene_name}_probes_against_transcript.xml"

    # Create blast command
    cmd_probes = [
        "blastn",
        "-query", fasta_path,
        "-subject", file_path,
        "-task", "blastn-short",
        "-outfmt", "5",
        "-out", probes_vs_trans_xml
    ]

    # Run blast and save the results
    try:
        print(f"Blasting the remaining probes against the transcript of interest...")
        result = subprocess.run(cmd_probes, capture_output=True, text=True, check=True)
        print("Finished!")
        print(result.stdout)
    except subprocess.CalledProcessError as e:
        print(f"Error during the blast search : {e.stderr}")

In [ ]:
# Check the results of the blast search
if blast_genomes == "Y":
    
    bad_probes = set()
    probe_hits = SearchIO.parse(probes_vs_trans_xml, "blast-xml")
    print(f"\n{'Probe':<25} | {'Alignment length':<12} | {'Identity':<8}")
    print("-" * 80)

    for qresult in probe_hits:
        probe_id = qresult.id
        for hit in qresult:
            for hsp in hit:
                p_start = hsp.hit_start
                p_end = hsp.hit_end
                p_ident = (hsp.ident_num / hsp.aln_span) * 100
                identity = (hsp.ident_num / hsp.aln_span) * 100
            
                # On vérifie si cette sonde chevauche une zone à risque du transcript
                for b_start, b_end in df_bad_regions[['Start_trans', 'End_trans']].itertuples(index=False):
                
                    # Calcul de l'intersection
                    overlap = min(p_end, b_end) - max(p_start, b_start)
                
                    if overlap >= 20 and p_ident >= 90:
                        bad_probes.add(probe_id)
                        print(f"{probe_id:<25} | {overlap:>3} bases    | {p_ident:>6.2f}%")
                        break

    print("-" * 80)
    print(f"\n Number of problematic probes: {len(bad_probes)}")
    print(bad_probes)

In [ ]:
# Decide to remove or to keep the probes
if blast_genomes == "Y":
    
    if not bad_probes:
        df_qc_blast_2 = df_qc_blast
        print("No problematic probes were found to map in the problematic regions of the transcript of interest.")
    else:
        print(f"{len(bad_probes)} probes align in the problematic regions of the transcript of interest: {bad_probes}")
    
        choice = input("\nDo you want to remove any probes? (Y/N): ").strip().upper()
    
        if choice == "Y":
            all_question = input("\nDo you want to exclude ALL detected bad probes? (Y/N): ").strip().upper()
        
            if all_question == "Y":
                to_exclude = list(bad_probes)
            else:
                print("Current bad probes:", bad_probes)
                custom_list = input("Enter the names of the probes to exclude (separated by space): ").strip()
                to_exclude = custom_list.split()

            # Filtrage du DataFrame
            # On suppose que l'index de df_qc_blast contient les noms (ex: OctVul6B003961T1_B1_PP_1)
            on_topic_indexes = [n for n, probe in enumerate(df_qc_blast.index.values) if probe not in to_exclude]
            df_qc_blast_2 = df_qc_blast.iloc[on_topic_indexes]
        
            print(f"\n{len(on_topic_indexes)} probes kept:\n", df_qc_blast_2.index.unique().values)
        
            # Mise à jour et export
            df_blast = pd.DataFrame(df_qc_blast_2.index.values, columns=["Identifier"])
            df_blast["Probe"] = df_qc_blast_2.blast.values
        
            # Sauvegarde TXT
            output_txt = f"output/{gene_name}_probes_updated.txt"
            df_blast.to_csv(output_txt, sep='\t', index=False, header=False, quoting=csv.QUOTE_NONE)
        
            # Sauvegarde FASTA
            probe_sequences = [SeqRecord(Seq(probe), id=str(idx)) for probe, idx in zip(df_blast.Probe.values, df_blast.Identifier.values)]
            fasta_path = f"output/{gene_name}_to_blast_against_sinensis.fasta"
        
            with open(fasta_path, "w") as output_handle:
                SeqIO.write(probe_sequences, output_handle, "fasta")
                
        else:
            print("No probes were removed.")

In [ ]:
# Align the transcript of interest to the O. sinensis genome for extra check
# Load database generated from O. sinensis genome (Here again you need to make sure you ran the custom_database_creation.ipynb script on the file)
if blast_genomes == "Y":

    # Define thresholds
    transcript_len = len(fullseq)
    min_overlap_len = 20
    min_probe_identity = 90.0

    # Define output file name
    blast_path = f"output/{gene_name}_blast_against_Osin_genome.xml"
    db_path = "GCF_006345805.1_ASM634580v1_rna.fna.gz"

    # Create blast command
    cmd_blastn = [
        "blastn",
        "-query", file_path,
        "-db", db_path,
        "-task", "blastn",
        "-evalue", "0.05",
        "-outfmt", "5",
        "-out", blast_path
    ]

    # Run blast and save the results
    try:
        print(f"Blasting the transcript of interest against the O. sinensis genome...")
        subprocess.run(cmd_blastn, capture_output=True, text=True, check=True)
        print("Finished!")
    except subprocess.CalledProcessError as e:
        print(f"Error during blast: {e.stderr}")

In [ ]:
# Look at the blast results
if blast_genomes == "Y":

    # Define function to fuse alignment intervals to calculate true alignment length
    def get_merged_coverage(hsp_list):
        """Fuse alignment intervals to calculate true alignment length."""
        if not hsp_list: return 0
        intervals = sorted([[h.query_start, h.query_end] for h in hsp_list])
        merged = []
        for curr_start, curr_end in intervals:
            if not merged or curr_start > merged[-1][1]:
                merged.append([curr_start, curr_end])
            else:
                merged[-1][1] = max(merged[-1][1], curr_end)
        return sum(end - start for start, end in merged)

    # Check alignment results
    qresults_sin = SearchIO.parse(blast_path, "blast-xml")
    summary_data = []

    for qresult in qresults_sin:
        for hit in qresult:
            total_coverage = get_merged_coverage(hit.hsps)
            total_ident_bases = sum(h.ident_num for h in hit.hsps)
            total_aln_span = sum(h.aln_span for h in hit.hsps)
            avg_identity = (total_ident_bases / total_aln_span * 100) if total_aln_span > 0 else 0
            query_starts = [h.query_start for h in hit.hsps]
            query_ends = [h.query_end for h in hit.hsps]
            summary_data.append({
                "O. sinensis transcript": hit.id,
                "Identity": avg_identity,
                "Total_Coverage": total_coverage,
                "Coverage_Ratio": total_coverage / transcript_len,
                "Start_trans": min(query_starts), 
                "End_trans": max(query_ends)
            })

    df_summary = pd.DataFrame(summary_data)

    print("\n--- Summary of regions where the transcript of interest is mapping ---")
    print("\n(Here you can check all the regions of the transcript that are mapping to the O. sinensis genome)\n")
    if not df_summary.empty:
        df_unique = (df_summary
                     .sort_values(by="Total_Coverage", ascending=False)
                     .drop_duplicates(subset=['Start_trans', 'End_trans'])
                     .head(20))
    
        print(df_unique.to_string(index=False))
        print(f"\nTotal unique mapping regions found: {len(df_unique)}")
    else:
        print("No hits found.")

In [ ]:
# Align the remaining probes to the transcript of interest to eliminate the ones mapping in bad regions of the transcript

if blast_genomes == "Y":

    # Define output file name
    probes_vs_trans_xml = f"output/{gene_name}_probes_against_transcript_step2.xml"

    # Create blast command
    cmd_probes = [
        "blastn",
        "-query", fasta_path,
        "-subject", file_path,
        "-task", "blastn-short",
        "-outfmt", "5",
        "-out", probes_vs_trans_xml
    ]

    # Run blast and save the results
    try:
        print(f"Blasting the remaining probes against the transcript of interest...")
        result = subprocess.run(cmd_probes, capture_output=True, text=True, check=True)
        print("Finished!")
        print(result.stdout)
    except subprocess.CalledProcessError as e:
        print(f"Error during the blast search : {e.stderr}")

In [ ]:
# Check the results of the blast search
if blast_genomes == "Y":

    # Define the thresholds for the selection of the corresponding transcript
    # (look at the best Identity percentage and Coverage_Ratio from the last table and adapt the numbers based on those)
    identity_threshold = int(input("Which identity threshold do you want to use? (number): \nThis will help define the best corresponding transcript from O. sinensis.\nLook at the best Identity percentage and Coverage_Ratio from the last table and adapt the number based on those.").strip())
    length_ratio_threshold = float(input("Which length ratio threshold do you want to use? (number): \nThis will help define the best corresponding transcript from O. sinensis.\nLook at the best Identity percentage and Coverage_Ratio from the last table and adapt the number based on those.").strip())
    
    def merge_regions(intervals):
        if not intervals:
            return []

        # 1. Trier par le début de chaque intervalle
        sort_intervals = sorted(intervals)
    
        # 2. Initialiser avec le premier intervalle
        merge = [sort_intervals[0]]
    
        for start, end in sort_intervals[1:]:
            last_start, last_end = merge[-1]
        
            # S'il y a un chevauchement (le début actuel est avant la fin du dernier)
            if start <= last_end:
                # On met à jour la fin du dernier intervalle si nécessaire
                new_max = max(last_end, end)
                merge[-1] = (last_start, new_max)
            else:
                # Pas de chevauchement, on ajoute l'intervalle tel quel
                merge.append((start, end))
            
        return merge

    probe_hits = SearchIO.parse(probes_vs_trans_xml, "blast-xml")
    bad_probes_2 = set()
    rows2=[]
    if not df_summary.empty:
        mask_valid = (df_summary['Identity'] >= identity_threshold) & \
                     (df_summary['Coverage_Ratio'] >= length_ratio_threshold)
        valid_targets = df_summary[mask_valid]['O. sinensis transcript'].unique().tolist()
        print(f"\nCorresponding regions to our transcript of interest in O. sinensis:\n{valid_targets}")
        qresults_sin = SearchIO.parse(blast_path, "blast-xml")
        off_target_regions = []
        rows3 = []
        for qresult in qresults_sin:
            for hit in qresult:
                if hit.id not in valid_targets:
                    for hsp in hit:
                        off_target_regions.append((hsp.query_start, hsp.query_end))

        merged_off_target_regions = merge_regions(off_target_regions)
        
        print("\nHere are the problematic regions in the transcript:")
        print(list(dict.fromkeys(merged_off_target_regions)))
        if off_target_regions:
            probe_hits = SearchIO.parse(probes_vs_trans_xml, "blast-xml")
        
            for qresult in probe_hits:
                probe_id = qresult.id
                for hit in qresult:
                    for hsp in hit:
                        p_start, p_end = hsp.hit_start, hsp.hit_end
                        p_ident = (hsp.ident_num / hsp.aln_span) * 100
                        p_length = hsp.query_end - hsp.query_start
                    
                        for b_start, b_end in off_target_regions:
                            overlap = min(p_end, b_end) - max(p_start, b_start)
                            if overlap >= min_overlap_len and p_ident >= min_probe_identity:
                                bad_probes_2.add(probe_id)
                                rows2.append({
                                    "probe": hsp.query_id,
                                    "Identity": p_ident,
                                    "Length_probe":p_length,
                                    "Start_trans": hsp.hit_start,
                                    "End_trans": hsp.hit_end
                                })
                                break

            df_all_hits2 = pd.DataFrame(rows2)
            if not df_all_hits2.empty:
                print(f"\nProbes mapping to the problematic regions of the transcript:")
                print(df_all_hits2[df_all_hits2['Length_probe'] >= 20].head(10).to_string())
            else:
                print(f"\nNo probe mapping to the problematic regions of the transcript.")
    else:
        print("No mapping found in Sinensis genome.")

In [ ]:
# Decide to remove or to keep the problematic probes
if blast_genomes == "Y":

    # --- 4. Conclusion et Nettoyage ---
    if not bad_probes_2:
        df_qc_blast_3 = df_qc_blast_2
        bad_probes_2 = bad_probes
        print("\nNo problematic probes detected.") 
    else:
        print(f"\n{len(bad_probes_2)} probes map to regions of the transcript mapping to the O. sinensis genome:\n{bad_probes_2}")
        choice = input("\nDo you want to remove any probes? (Y/N): ").strip().upper()
    
        if choice == "Y":
            all_q = input("\nExclude ALL? (Y/N): ").strip().upper()
            to_exclude = list(bad_probes_2) if all_q == "Y" else input("Enter probe names: ").split()
        
            # df_qc_blast_2 doit exister au préalable dans ton notebook
            on_topic_indexes = [n for n, probe in enumerate(df_qc_blast_2.index.values) if probe not in to_exclude]
            df_qc_blast_3 = df_qc_blast_2.iloc[on_topic_indexes]
            print(f"\n{len(on_topic_indexes)} probes kept:\n", df_qc_blast_3.index.unique().values)
        
            # Export final
            output_txt = f"output/{gene_name}_probes_updated.txt"
            df_export = pd.DataFrame({"Identifier": df_qc_blast_3.index, "Probe": df_qc_blast_3.blast})
            df_export.to_csv(output_txt, sep='\t', index=False, header=False, quoting=csv.QUOTE_NONE)
        else:
            df_qc_blast_3 = df_qc_blast_2
            bad_probes_2 = bad_probes

In [ ]:
# Generate IDT order form
if len(bad_probes_2) > 0:
    current_n_probes = len(df_qc_blast_3) 
    data_list = []
    
    for name, row in df_qc_blast_3.iterrows():
        name_parts = str(name).split("_")
        pool_name = "_".join(name_parts[:-2]) + f"_{current_n_probes}PP"
    
        # Ajouter le premier oligo à la liste
        data_list.append({
            "Pool name": pool_name,
            "Sequence": row.Initiator + row.Spacer.upper() + row.Probe
        })
    
        # Ajouter le deuxième oligo à la liste
        data_list.append({
            "Pool name": pool_name,
            "Sequence": row["Probe.1"] + row["Spacer.1"].upper() + row["Initiator.1"]
        })
    df_idt = pd.DataFrame(data_list)

    print(f"Generated an order of {len(df_qc_blast_3)} probe pairs ({len(df_idt)} oligos)")
    print(df_idt)
else:
    data_list = []

    for name, row in df_qc_blast.iterrows():
        # Note: si 'name' est l'index, utilisez 'name', sinon utilisez 'row.name'
        name_parts = str(name).split("_")
        pool_name = "_".join(name_parts[:-2]) + f"_{n_probes}PP"
    
        # Ajouter le premier oligo à la liste
        data_list.append({
            "Pool name": pool_name,
            "Sequence": row.Initiator + row.Spacer.upper() + row.Probe
        })
    
        # Ajouter le deuxième oligo à la liste
        data_list.append({
            "Pool name": pool_name,
            "Sequence": row["Probe.1"] + row["Spacer.1"].upper() + row["Initiator.1"]
        })
    df_idt = pd.DataFrame(data_list)

    print(f"Generated an order of {len(df_qc_blast)} probe pairs ({len(df_idt)} oligos)")
    print(df_idt)

In [ ]:
# Export order form

idt_path = f"output/{gene_name}_idt_order.xlsx"

df_idt.to_excel(idt_path, index=False)